# UdaPlay — Part 1: RAG Pipeline

This notebook builds the internal knowledge base for UdaPlay:
1. Load and validate the raw video game JSON data.
2. Convert it into embedding-ready `Document` objects.
3. Build a FAISS vector store with OpenAI embeddings and persist it to disk.
4. Run verification similarity-search queries.


## 0. Setup: environment variables & logging

In [ ]:
import sys
from pathlib import Path

# Allow imports from the project's src/ package when running from notebooks/
sys.path.insert(0, str(Path.cwd().parent))

from src.config import verify_environment, GAMES_DATA_PATH, FAISS_INDEX_PATH
from src.logging_config import get_logger

verify_environment()  # raises a clear error if OPENAI_API_KEY / TAVILY_API_KEY are missing
logger = get_logger("notebook_01")
logger.info("Environment verified. Ready to build the FAISS index.")


## 1. Load & validate the raw game data

In [ ]:
from src.rag.data_loader import load_game_records, records_to_documents

records = load_game_records(GAMES_DATA_PATH)
print(f"Loaded {len(records)} validated game records")
records[0]


## 2. Convert records into embedding-ready Documents

In [ ]:
documents = records_to_documents(records)
print(documents[0].page_content)
print(documents[0].metadata)


## 3. Build the FAISS vector store and persist it to disk

In [ ]:
from src.rag.vector_store import build_vector_store

vector_store = build_vector_store(data_path=GAMES_DATA_PATH, index_path=FAISS_INDEX_PATH, persist=True)
print(f"FAISS index built and saved to: {FAISS_INDEX_PATH}")


## 4. Reload from disk (sanity check persistence works)

In [ ]:
from src.rag.vector_store import load_vector_store

reloaded_store = load_vector_store(FAISS_INDEX_PATH)
print("Reloaded index vector count:", reloaded_store.index.ntotal)


## 5. Semantic Retrieval Verification

Run a few test queries and inspect the top-k similarity results and scores.

In [ ]:
from src.rag.vector_store import similarity_search

test_queries = [
    "When was Pokémon Red launched, and on what platform?",
    "open world western game with an outlaw protagonist",
    "puzzle game involving portals",
]

for q in test_queries:
    print("=" * 80)
    print("QUERY:", q)
    results = similarity_search(q, k=3)
    for doc, score in results:
        print(f"  - {doc.metadata['Name']} ({doc.metadata['Platform']}, {doc.metadata['ReleaseYear']}) "
              f"score={score:.3f}")


Part 1 complete: the FAISS index is built, persisted at `faiss_index_udaplay/`, and verified with top-k similarity search. Continue to `Udaplay_02_solution_project.ipynb` for the multi-agent workflow.